In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

token = user_secrets.get_secret("vivqa")

username = "nmluan85"
repo_name = "ViVQA-LanguageBias"

repo_url = f"https://{token}@github.com/{username}/{repo_name}.git"
!git clone {repo_url}

Cloning into 'ViVQA-LanguageBias'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 90 (delta 34), reused 78 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 88.56 KiB | 1.81 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
!pip install -r /kaggle/working/ViVQA-LanguageBias/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 8.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 kB 14.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [3]:
!pip install torchscale
!pip install mlflow efficientnet_pytorch underthesea
!pip install timm==1.0.13

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.3/53.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.7 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 0.6.13
    Uninstalling timm-0.6.13:
      Successfully uninstalled timm-0.6.13
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchscale 0.3.0 requires timm==0.6.13, but you have timm 1.0.13 which is incompatible.


In [4]:
import sys
import os

repo_path = "/kaggle/working/ViVQA-LanguageBias"
if repo_path not in sys.path:
    sys.path.append(repo_path)

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use only the first GPU
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'mlflow-vivqa'

In [5]:
import argparse
base_path = "/kaggle/working/ViVQA-LanguageBias"
opt = argparse.Namespace()

# Training Parameters
opt.output_dir = "./output"
opt.log_level = "passive"
opt.lr_scheduler_type = "cosine"
opt.warmup_ratio = 0.1
opt.logging_strategy = "epoch"
opt.save_strategy = "epoch"
opt.save_total_limit = 1
opt.train_batch_size = 16
opt.eval_batch_size = 16
opt.epochs = 2
opt.learning_rate = 3e-5
opt.weight_decay = 0.01
opt.workers = 2

# Dataset Paths (Absolute Kaggle Paths)
opt.image_path = f"/kaggle/input/datasets/luan85/viet-vqa/images"
opt.ans_path = f"/kaggle/input/datasets/luan85/viet-vqa/vocab.json"
opt.train_path = f"/kaggle/input/datasets/luan85/viet-vqa/train.csv"
opt.val_path = f"{base_path}/data/ViVQA-csv/val.csv"
opt.test_path = f"/kaggle/input/datasets/luan85/viet-vqa/test.csv"
opt.feature_paths = f"/kaggle/working/"

# Model Architecture
opt.drop_path_rate = 0.3
opt.encoder_layers = 6
opt.encoder_attention_heads_layers = 6
opt.classes = 353

In [6]:
import timm
import torch
import torch.nn as nn
import numpy as np
import cv2
from efficientnet_pytorch import EfficientNet
from omegaconf import OmegaConf
from lavis.models.blip2_model.blip2_qformer import Blip2Qformer
from lavis.processors.base_processor import BaseProcessor
from lavis.processors.blip_processor import (
    BlipCaptionProcessor, 
    BlipQuestionProcessor,
    BlipImageTrainProcessor,
    BlipImageEvalProcessor, 
    Blip2ImageTrainProcessor,
)

class EfficientnetExtractor(nn.Module):
    def __init__(self, model_name):
        super(EfficientnetExtractor, self).__init__()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = EfficientNet.from_pretrained(model_name, advprop=True).to(self.device)
        self.model_name = model_name
        self.processor = self.load_preprocess()

    def load_preprocess(self):
        config = OmegaConf.load(Blip2Qformer.default_config_path(model_type="pretrain"))
        preprocess_cfg = config.preprocess

        def _build_proc_from_cfg(cfg):
            if cfg is None:
                return BaseProcessor()
            if cfg.name == "blip_caption":
                return BlipCaptionProcessor.from_config(cfg)
            elif cfg.name == "blip_question":
                return BlipQuestionProcessor.from_config(cfg)
            elif cfg.name == "blip_image_train":
                return BlipImageTrainProcessor.from_config(cfg)
            elif cfg.name == "blip_image_eval":
                return BlipImageEvalProcessor.from_config(cfg)
            elif cfg.name == "blip2_image_train":
                return Blip2ImageTrainProcessor.from_config(cfg)

        vis_proc_cfg = preprocess_cfg.get("vis_processor")
        vis_eval_cfg = vis_proc_cfg.get("eval")
        vis_processors = _build_proc_from_cfg(vis_eval_cfg)
        return vis_processors
    
    def forward(self, *images):
        self.model.eval()
        images_transformed = torch.stack([self.processor(image.convert('RGB')).to(self.device) for image in images])
        
        x = self.model.extract_features(images_transformed)  # [B, C=2560, H=7, W=7]
        x = nn.AdaptiveAvgPool2d((7,7))(x)   
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, H*W, C)  # [B, 49, 2560]
        
        return x

2026-03-22 11:23:29.951674: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774178610.317928      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774178610.419691      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774178611.389496      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774178611.389540      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774178611.389543      24 computation_placer.cc:177] computation placer alr

In [7]:
import torch
from torch.utils.data import DataLoader
from PIL import Image
import os
import glob
from tqdm import tqdm
import numpy as np
import h5py


def extract_features(vs_encoder, opt, model_name):
    file_paths = sorted(glob.glob(os.path.join(opt.image_path, '*.jpg')))
    features_shape = (len(file_paths), 49, 2560)

    with h5py.File(f'{opt.feature_paths}/{model_name}.hdf5', 'w') as f:
        features = f.create_dataset('features', shape=features_shape, dtype='float16')
        img_ids = f.create_dataset('ids', shape=(len(file_paths),), dtype='int32')

        j = k = 0
        data_loader = DataLoader(file_paths, batch_size=132, shuffle=False, num_workers=4)
        for paths in tqdm(data_loader):
            ids = [os.path.basename(path).split('.')[0] for path in paths]
            imgs = [Image.open(path) for path in paths]
            
            with torch.no_grad():
                out = vs_encoder(*imgs)
                
            k = j + len(paths)
            features[j:k] = out.cpu().numpy().astype('float16')
            img_ids[j:k] = np.array(ids, dtype='int32')
            j = k

In [8]:
model_efficientnet = EfficientnetExtractor("efficientnet-b7")
extract_features(model_efficientnet, opt, "efficientnet-b7")

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/adv-efficientnet-b7-4652b6dd.pth" to /root/.cache/torch/hub/checkpoints/adv-efficientnet-b7-4652b6dd.pth


100%|██████████| 255M/255M [00:01<00:00, 245MB/s]


Loaded pretrained weights for efficientnet-b7


100%|██████████| 79/79 [09:13<00:00,  7.01s/it]
